In [1]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install litellm
%pip install -U crewai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import requests
import litellm
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool
from dotenv import load_dotenv

loaded = load_dotenv()

# Configure LiteLLM
litellm.drop_params = True

# Monkey patch litellm.completion to handle parameter mapping
original_completion = litellm.completion

def patched_completion(*args, **kwargs):
    # If max_tokens is present and max_completion_tokens is not, map it
    if 'max_tokens' in kwargs and 'max_completion_tokens' not in kwargs:
        kwargs['max_completion_tokens'] = kwargs.pop('max_tokens')
    
    return original_completion(*args, **kwargs)

# Apply the patch
litellm.completion = patched_completion

In [4]:
# Tavily API Key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload)
        data = response.json()

        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)

search_tool = TavilySearchTool()

# ---- Azure LLM - FIXED ----
# Using max_tokens (not max_completion_tokens) with the monkey patch
llm = LLM(
    model=f"azure/{os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    is_litellm=True,
    temperature=1, # some models don't support temperature
    max_tokens=3500  # This will be converted to max_completion_tokens
)

In [6]:
# Researcher agent
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for FMCG",
    backstory="You are an expert in artificial intelligence and stay updated with the latest research trends in FMCG.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=1,
    tools=[search_tool]
)

# Writer agent
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory="You are an experienced technical writer with expertise in summarizing research for executives.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)

In [7]:
# ---- Tasks ----
task_research = Task(
    description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
    expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
    agent=researcher
)

task_write = Task(
    description="""
Write a concise executive summary using the research notes.

Requirements:
- Maximum 100 words
- Use bullet points
- Focus only on the 3 key advancements
""",
    expected_output="Executive summary of AI advancements in FMCG.",
    agent=writer,
    context=[task_research]
)

In [8]:
# ---- Crew ----
crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

result = await crew.kickoff_async()

print("\nFinal Output:\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 43c77e5c-fc07-4053-9f0a-469c88e6ab56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the web and identify the top 3 recent advancements in AI for FMCG.                                │
│  ID: a9527a86-728a-480d-b275-2d6fbb1ec480                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Search the web and identify the top 3 recent advancements in AI for FMCG.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {"query": "recent advancements AI FMCG 2023 2024 demand forecasting shelf analytics generative AI        │
│  product development personalization retail examples companies"}                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: web_search                                                                                               │
│  Output: AI Retail 2026: How Artificial Intelligence Transforms ... -                                           │
│  https://www.articsledge.com/post/ai-retail                                                                     │
│  From Personalization to Prediction: How AI and Customer Intent are Reimagining Retail | RampUp 2026 -          │
│  https://www.youtube.com/watch?v=X8mkkWhzvM0                                                                    │
│  Artificial Intelligence in FMCG and Retail Market -                                                            │
│  https://dataintelo.com/report/global-artificial-intelligence-in-fmcg-and-retail-market                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - AI demand forecasting & supply‑chain optimization: ML platforms (Blue Yonder, Relex, o9; PepsiCo/Unilever    │
│  pilots) improve forecast accuracy, reduce stockouts and inventory costs, and enable scenario planning.         │
│  - Computer‑vision shelf analytics & in‑store automation: Solutions (Trax, Scandit, Bossa Nova, Amazon Go)      │
│  detect out‑of‑stocks, enforce planograms and speed restocking, lifting on‑shelf availability and execution.    │
│  - Generative AI for personalization & product innovation: Generative models (used by Unilever, P&G,            │
│  Coca‑Cola) accelerate NPD, create hyper‑personalized marketing content, and compress concept‑to‑market         │
│  cycles.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Write a concise executive summary using the research notes.                                                    │
│                                                                                                                 │
│  Requirements:                                                                                                  │
│  - Maximum 100 words                                                                                            │
│  - Use bullet points                                                                                            │
│  - Focus only on the 3 key advancements                                                                         │
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 43c77e5c-fc07-4053-9f0a-469c88e6ab56                                                                       │
│  Final Output: - AI demand forecasting & supply‑chain optimization: ML platforms (Blue Yonder, Relex, o9;       │
│  PepsiCo/Unilever pilots) improve forecast accuracy, reduce stockouts and inventory costs, and enable scenario  │
│  planning.                                                                                                      │
│  - Computer‑vision shelf analytics & in‑store automation: Solutions (Trax, Scandit, Bossa Nova, Amazon Go)      │
│  detect out‑of‑stocks, enforce planograms and speed restocking, lifting on‑shelf availability and execution.    │
│  - Generative AI for personalization & product innovation: Generative models (used by Unilever, P&G,            │
│  Coca‑Cola) accelerate NPD, create hyper‑personalized marketing content, and compress concept‑to‑market         │
│  cycles.                                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final Output:

- AI demand forecasting & supply‑chain optimization: ML platforms (Blue Yonder, Relex, o9; PepsiCo/Unilever pilots) improve forecast accuracy, reduce stockouts and inventory costs, and enable scenario planning.  
- Computer‑vision shelf analytics & in‑store automation: Solutions (Trax, Scandit, Bossa Nova, Amazon Go) detect out‑of‑stocks, enforce planograms and speed restocking, lifting on‑shelf availability and execution.  
- Generative AI for personalization & product innovation: Generative models (used by Unilever, P&G, Coca‑Cola) accelerate NPD, create hyper‑personalized marketing content, and compress concept‑to‑market cycles.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯